In [1]:
# Install packages
!pip install pybiomart pandas openpyxl biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00


In [ ]:
# Import libraries
import pandas as pd
import os
import time
from pybiomart import Server
from Bio import Entrez

# NCBI require email
Entrez.email = "2020ict96@stu.vau.ac.lk"

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
# Load Step 1 output
df = pd.read_csv('gwas_snps_combined.csv')

print(f"Loaded Step 1 data: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns : {list(df.columns)}")
df.head()

Loaded Step 1 data: 5860 rows, 3 columns
Columns : ['rsId', 'functionalClass', 'Disease']


,rsId,functionalClass,Disease
0,rs77212406,regulatory_region_variant,AD
1,rs115795926,intergenic_variant,AD
2,rs1761608,non_coding_transcript_exon_variant,AD
3,rs150180355,intergenic_variant,AD
4,rs111609130,intron_variant,AD


In [ ]:
# Extract clean rsID list
rsid_list = df['rsId'].dropna().unique().tolist()

print(f"Total unique rsIDs to map : {len(rsid_list)}")
print(f"Sample rsIDs              : {rsid_list[:5]}")

Total unique rsIDs to map : 5852
Sample rsIDs              : ['rs77212406', 'rs115795926', 'rs1761608', 'rs150180355', 'rs111609130']


In [ ]:
# Connect to Ensembl BioMart (GRCh38)
print("Connecting to Ensembl BioMart (GRCh38) ...")

server  = Server(host='http://www.ensembl.org')
mart    = server['ENSEMBL_MART_SNP']
dataset = mart['hsapiens_snp']

print( "Connected — dataset: hsapiens_snp (Human SNPs, GRCh38)")

Connecting to Ensembl BioMart (GRCh38) ...
Connected — dataset: hsapiens_snp (Human SNPs, GRCh38)


In [ ]:
# Query BioMart in batches of 500
print("=== Querying BioMart for SNP mapping ===\n")

BATCH_SIZE  = 500
all_results = []

for i in range(0, len(rsid_list), BATCH_SIZE):
    batch      = rsid_list[i : i + BATCH_SIZE]
    batch_num  = (i // BATCH_SIZE) + 1
    total_batches = (len(rsid_list) // BATCH_SIZE) + 1
    print(f"   Batch {batch_num}/{total_batches} — querying {len(batch)} rsIDs ...")

    try:
        result = dataset.query(
            attributes=[
                'refsnp_id',
                'chr_name',
                'chrom_start',
                'consequence_type_tv',
                'ensembl_gene_stable_id',
                'ensembl_transcript_stable_id',
                'associated_gene',
            ],
            filters={'snp_filter': batch}
        )
        if result is not None and not result.empty:
            all_results.append(result)
            print(f"      → {len(result)} records returned")
        else:
            print(f"      → No records returned")
    except Exception as e:
        print(f"      Error in batch {batch_num}: {str(e)[:100]}")

print("\n BioMart query complete")

=== Querying BioMart for SNP mapping ===

   Batch 1/12 — querying 500 rsIDs ...
      → 430 records returned
   Batch 2/12 — querying 500 rsIDs ...
      → 499 records returned
   Batch 3/12 — querying 500 rsIDs ...
      Error in batch 3: Query ERROR: caught BioMart::Exception::Database: Could not connect to mysql database snp_mart_115: 
   Batch 4/12 — querying 500 rsIDs ...
      → 533 records returned
   Batch 5/12 — querying 500 rsIDs ...
      → 503 records returned
   Batch 6/12 — querying 500 rsIDs ...
      → 518 records returned
   Batch 7/12 — querying 500 rsIDs ...
      → 542 records returned
   Batch 8/12 — querying 500 rsIDs ...
      Error in batch 8: Query ERROR: caught BioMart::Exception::Database: Could not connect to mysql database snp_mart_115: 
   Batch 9/12 — querying 500 rsIDs ...
      Error in batch 9: Query ERROR: caught BioMart::Exception::Database: Could not connect to mysql database snp_mart_115: 
   Batch 10/12 — querying 500 rsIDs ...
      → 509 record

In [ ]:
# Combine BioMart results & rename columns
mapped = pd.concat(all_results, ignore_index=True)
mapped.columns = [
    'rsId', 'chromosome', 'position',
    'consequence_type', 'ensembl_gene_id',
    'ensembl_transcript_id', 'gene_symbol'
]

print(f"Total mapped records : {len(mapped)}")
print(f"Unique rsIDs mapped  : {mapped['rsId'].nunique()}")
mapped.head()

Total mapped records : 4896
Unique rsIDs mapped  : 3428


,rsId,chromosome,position,consequence_type,ensembl_gene_id,ensembl_transcript_id,gene_symbol
0,rs340849,1,213944747,NaN,NaN,NaN,PROX1
1,rs340849,1,213944747,NaN,NaN,NaN,intergenic
2,rs1340425,1,73920165,NaN,NaN,NaN,Intergenic
3,rs3818361,1,207611623,intron_variant,ENSG00000203710,ENST00000367049,CR1
4,rs3818361,1,207611623,intron_variant,ENSG00000203710,ENST00000367049,NaN


In [ ]:
#  Check consequence types
print("Consequence types found:")
print(mapped['consequence_type'].value_counts().to_string())

Consequence types found:
consequence_type
intron_variant                         1767
missense_variant                        189
3_prime_UTR_variant                      86
synonymous_variant                       55
5_prime_UTR_variant                      14
splice_region_variant                    14
splice_polypyrimidine_tract_variant       8
stop_gained                               3
frameshift_variant                        2
inframe_deletion                          1
inframe_insertion                         1


In [ ]:
# Filter: keep only 3′UTR variants
utr3 = mapped[mapped['consequence_type'] == '3_prime_UTR_variant'].copy()

print(f"SNPs in 3′UTR   : {len(utr3)}")
print(f"Unique rsIDs    : {utr3['rsId'].nunique()}")
print(f"Unique genes    : {utr3['gene_symbol'].nunique()}")
utr3.head()

SNPs in 3′UTR   : 86
Unique rsIDs    : 57
Unique genes    : 38


,rsId,chromosome,position,consequence_type,ensembl_gene_id,ensembl_transcript_id,gene_symbol
8,rs182798940,1,43453815,3_prime_UTR_variant,ENSG00000198198,ENST00000634258,Intergenic
19,rs139586416,1,226231828,3_prime_UTR_variant,ENSG00000183814,ENST00000681046,NaN
74,rs75582018,3,190428661,3_prime_UTR_variant,ENSG00000198398,ENST00000354905,NaN
381,rs201269048,19,45070855,3_prime_UTR_variant,ENSG00000104859,ENST00000221455,APOE
382,rs201269048,19,45070855,3_prime_UTR_variant,ENSG00000104859,ENST00000221455,NaN


In [ ]:
# Fetch disease-relevant genes from NCBI Gene database

disease_queries = {
    'AD' : 'Alzheimer disease[disease/phenotype associated] AND Homo sapiens[organism]',
    'PD' : 'Parkinson disease[disease/phenotype associated] AND Homo sapiens[organism]',
    'ALS': 'amyotrophic lateral sclerosis[disease/phenotype associated] AND Homo sapiens[organism]',
}

print("=== Fetching disease-relevant genes from NCBI Gene database ===\n")

disease_genes = {}

for disease, query in disease_queries.items():
    print(f"Processing {disease} ...")
    try:
        # Step A: Search NCBI Gene — get gene IDs
        search_handle  = Entrez.esearch(db='gene', term=query, retmax=500, usehistory='y')
        search_results = Entrez.read(search_handle)
        search_handle.close()

        gene_ids = search_results['IdList']
        count    = search_results['Count']
        print(f"   Total hits in NCBI Gene : {count}")
        print(f"   Gene IDs retrieved      : {len(gene_ids)}")

        if not gene_ids:
            disease_genes[disease] = []
            continue

        # Step B: Fetch XML summaries → extract gene symbols
        fetch_handle = Entrez.efetch(
            db='gene',
            id=','.join(gene_ids),
            rettype='gene_table',
            retmode='text'
        )
        fetch_text = fetch_handle.read()
        fetch_handle.close()

        # Step C: Parse gene symbols from the gene_table text output
        # Format: Symbol \t GeneID \t Status \t ...
        symbols = []
        for line in fetch_text.strip().split('\n'):
            line = line.strip()
            if not line or line.startswith('#') or line.startswith('tax_id'):
                continue
            parts = line.split('\t')
            # gene_table columns: tax_id | GeneID | Symbol | LocusTag | ...
            if len(parts) >= 3:
                symbol = parts[2].strip()  # Symbol is column index 2
                if symbol and symbol != '-' and not symbol.startswith('LOC'):
                    symbols.append(symbol)

        disease_genes[disease] = list(set(symbols))
        print(f"   Gene symbols extracted  : {len(disease_genes[disease])}")
        print(f"   Sample genes            : {sorted(disease_genes[disease])[:10]}")

        time.sleep(0.4)  # Respect NCBI rate limit (max 3 req/sec)

    except Exception as e:
        print(f"   Error for {disease}: {e}")
        disease_genes[disease] = []

print("\n Gene fetching complete")

=== Fetching disease-relevant genes from NCBI Gene database ===

Processing AD ...
   Total hits in NCBI Gene : 48
   Gene IDs retrieved      : 48
   Gene symbols extracted  : 809
   Sample genes            : ['1-128', '1-144', '1-165', '1-170', '1-174', '1-205', '1-236', '1-253', '1-261', '1-32']
Processing PD ...
   Total hits in NCBI Gene : 57
   Gene IDs retrieved      : 57
   Gene symbols extracted  : 1297
   Sample genes            : ['1-105', '1-1108', '1-1218', '1-122', '1-133', '1-1485', '1-165', '1-167', '1-173', '1-184']
Processing ALS ...
   Total hits in NCBI Gene : 183
   Gene IDs retrieved      : 183
   Gene symbols extracted  : 1590
   Sample genes            : ['1-104', '1-110', '1-111', '1-114', '1-118', '1-131', '1-158', '1-160', '1-161', '1-163']

 Gene fetching complete


In [ ]:
#  Merge with disease labels from Step 1
utr3_merged = utr3.merge(
    df[['rsId', 'Disease']].drop_duplicates(),
    on='rsId',
    how='left'
)

print(f"3′UTR SNPs with disease labels : {len(utr3_merged)}")
print("\nBreakdown by disease:")
print(utr3_merged['Disease'].value_counts().to_string())

3′UTR SNPs with disease labels : 95

Breakdown by disease:
Disease
AD     59
PD     33
ALS     3


In [ ]:
# Review what genes were fetched per disease
for disease, genes in disease_genes.items():
    print(f"{disease} → {len(genes)} genes: {sorted(genes)[:15]} ...")

AD → 809 genes: ['1-128', '1-144', '1-165', '1-170', '1-174', '1-205', '1-236', '1-253', '1-261', '1-32', '1-34', '1-348', '1-362', '1-46', '1-545'] ...
PD → 1297 genes: ['1-105', '1-1108', '1-1218', '1-122', '1-133', '1-1485', '1-165', '1-167', '1-173', '1-184', '1-229', '1-2383', '1-242', '1-25', '1-273'] ...
ALS → 1590 genes: ['1-104', '1-110', '1-111', '1-114', '1-118', '1-131', '1-158', '1-160', '1-161', '1-163', '1-199', '1-211', '1-214', '1-221', '1-241'] ...


In [ ]:
# Flatten to one set of all disease-relevant genes
all_disease_genes = set(g for genes in disease_genes.values() for g in genes)

print(f"Total unique disease-relevant genes fetched from NCBI : {len(all_disease_genes)}")

Total unique disease-relevant genes fetched from NCBI : 3653


In [ ]:
# Filter 3′UTR SNPs to disease-relevant genes only
utr3_filtered = utr3_merged[
    utr3_merged['gene_symbol'].isin(all_disease_genes)
].copy()

print(f"3′UTR SNPs in disease-relevant genes : {len(utr3_filtered)}")
print(f"Unique rsIDs                          : {utr3_filtered['rsId'].nunique()}")
print(f"Unique genes                          : {utr3_filtered['gene_symbol'].nunique()}")
print("\nGenes found:")
print(utr3_filtered['gene_symbol'].value_counts().to_string())

3′UTR SNPs in disease-relevant genes : 0
Unique rsIDs                          : 0
Unique genes                          : 0

Genes found:
Series([], )
